# 设备名称匹配

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 配置matplotlib中文显示
import platform
from rapidfuzz import fuzz, process
from sklearn.cluster import AgglomerativeClustering
system_name = platform.system()

if system_name == 'Windows':
    # Windows系统
    plt.rcParams['font.family'] = ['SimHei']
elif system_name == 'Darwin':
    # macOS系统
    plt.rcParams['font.family'] = ['PingFang HK']
elif system_name == 'Linux':
    # Linux系统（可能需要根据具体发行版进行调整）
    plt.rcParams['font.family'] = ['DejaVu Sans']
else:
    # 其他系统或无法识别系统，默认字体
    plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 导入数据

## 油耗表

In [2]:
# import data
file_path = 'Resources/Fuel_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['油耗信息', '设备信息']


In [11]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,班次,设备名称,设备编号,油品种类,油品消耗
0,2019-08-04,Night,TEREX TR100#56,HT0056,IC IC,476
1,2019-08-04,Night,TEREX TR100#58,HT0058,IC IC,462
2,2019-08-04,Night,TEREX TR100#59,HT0059,IC IC,451
3,2019-08-04,Night,TEREX TR100#60,HT0060,IC IC,454
4,2019-08-04,Night,TEREX TR100#61,HT0061,IC IC,502
...,...,...,...,...,...,...
222747,2026-04-30,Night,\nTOYOTA LANDCRUISER78 LV#1225,LV1225,Золотой /300007/,60
222748,2026-04-30,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,LV1226,Золотой /300007/,30.84
222749,2026-04-30,Night,North Benz ST#137 /ST0137/,ST0137,Золотой /300007/,173.35
222750,2026-04-30,Night,North Benz ST#141,ST141,Золотой /300007/,350


In [12]:
print(df_sheet1["油品消耗"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["油品消耗"])
print(df_sheet1["油品消耗"].isnull().sum())

1
0


In [13]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
df_sheet1.drop_duplicates(inplace=True)

In [14]:
match_table = pd.DataFrame()
match_table['设备名称'] = df_sheet1['设备名称']
match_table['设备编号'] = df_sheet1['设备编号']
match_table['公司'] = np.nan
match_table['来源'] = f"Fuel_{xlsx.sheet_names[0]}"
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
219252,AS4006 LP#0075,LP75,NaN,Fuel_油耗信息
219642,AS4006 LP#0074,LP74,NaN,Fuel_油耗信息
219644,4931ХОҮ/ LV#1232 /JMC / JX1033/,LV#1232,NaN,Fuel_油耗信息
220201,ZTC250A ME#136,ME#136,NaN,Fuel_油耗信息


In [15]:
df_sheet1 = pd.read_excel(file_path, sheet_name=1)
df_sheet1

,日期,班次,设备名称,设备编号,发动机小时数开始,发动机小时数结束,运行小时数
0,2019-09-01,Day,TEREX TR100#58,HT0058,301.1,311,9.9
1,2019-09-01,Day,TEREX TR100#64,HT0064,307.7,307.7,0
2,2019-09-01,Day,NHL NTE240 #66,HT0066,64,64,0
3,2019-09-01,Day,NHL NTE240 #67,HT0067,62,62,0
4,2019-09-01,Day,NHL NTE240 #68,HT0068,66,66,0


In [16]:
print(df_sheet1["设备名称"].isnull().sum())
# df_sheet1.dropna(inplace=True, subset=["油品消耗"])
# print(df_sheet1["油品消耗"].isnull().sum())

62


In [17]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
df_sheet1['来源'] = f"Fuel_{xlsx.sheet_names[1]}"
df_sheet1.drop_duplicates(inplace=True)

In [18]:
match_table = pd.concat([match_table, df_sheet1[['设备名称', '设备编号', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
599234,AS4006 LP#0074,LP74,NaN,Fuel_设备信息
599235,AS4006 LP#0075,LP75,NaN,Fuel_设备信息
599242,4931ХОҮ/ LV#1232 /JMC / JX1033/,LV#1232,NaN,Fuel_设备信息
604475,ZTC250A ME#136,ME#136,NaN,Fuel_设备信息


## 电力表

In [19]:
# import data
file_path = 'Resources/电力_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['电力消耗']


In [20]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,设备名称,电力消耗
0,2023-02-01,EX-5600#111,20873.9552
1,2023-02-01,EX-5600#113,26296.6428
2,2023-02-02,EX-5600#111,18047.4620
3,2023-02-02,EX-5600#113,25303.8492
4,2023-02-03,EX-5600#113,26525.0880
...,...,...,...
5262,2026-04-30,EX-5600#113,24600.0000
5263,2026-04-30,EX-5600#122,26892.0000
5264,2026-04-30,EX-3600#112,20268.0000
5265,2026-04-30,EX-5600#111,26568.0000


In [21]:
print(df_sheet1["电力消耗"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["电力消耗"])
print(df_sheet1["电力消耗"].isnull().sum())

0
0


In [22]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = np.nan
df_sheet1['来源'] = f"电力_{xlsx.sheet_names[0]}"
df_sheet1.drop_duplicates(inplace=True)

In [23]:
match_table = pd.concat([match_table, df_sheet1[['设备名称', '设备编号', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
1123,EX-5600#111,NaN,NaN,电力_电力消耗
1124,EX-5600#113,NaN,NaN,电力_电力消耗
1131,EX-5600#122,NaN,NaN,电力_电力消耗
1204,EX-5600#123,NaN,NaN,电力_电力消耗


## 产量

In [24]:
# import data
file_path = 'Resources/产量_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['生产数据', '运行数据']


In [25]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1

,日期,班次,矿卡名称,挖机名称,矿石类型,数量,产量
0,2019-08-17,Day,TEREX TR100 #56,CAT 390D #02,Хө,4.0,128.0
1,2019-08-17,Day,TEREX TR100 #57,CAT 390D #02,Хө,4.0,128.0
2,2019-08-17,Day,TEREX TR100 #58,CAT 390D #02,Хө,3.0,96.0
3,2019-08-17,Day,TEREX TR100 #59,CAT 390D #02,Хө,4.0,128.0
4,2019-08-17,Day,TEREX TR100 #60,CAT 390D #02,Хө,5.0,160.0
...,...,...,...,...,...,...,...
329639,2026-04-30,Night,NHL NTE240AC #1220,EX5600-6 #113,Хө,19.0,1615.0
329640,2026-04-30,Night,NHL NTE240AC #1221,EX5600-6 #113,Хө,19.0,1615.0
329641,2026-04-30,Night,NHL NTE240AC #1221,EX2600-6 #121,Хө,1.0,85.0
329642,2026-04-30,Night,NHL NTE240AC #1222,EX5600-6 #113,Хө,18.0,1530.0


In [29]:
data_to_process = df_sheet1[(df_sheet1["产量"]==0)&(df_sheet1["数量"]!=0)]
data_to_process

,日期,班次,矿卡名称,挖机名称,矿石类型,数量,产量
5107,2019-10-27,Night,KOMATSU 785 #05KH,Hitachi 5600#11 /CHU/,Хө 装载数,2.0,0.0
5108,2019-10-27,Night,KOMATSU 785 #05KH,PC 2000 #01KH,Хө,13.0,0.0
8482,2019-11-23,Day,HITACHI 4000#01KH,Hitachi 5600#13 /CHU/ 装载数,Хө,3.0,0.0
8483,2019-11-23,Day,HITACHI 4000#01KH,HITACHI EX2600E #03KH,Хө,7.0,0.0
8484,2019-11-23,Day,HITACHI 4000#01KH,LBR 9250 #03KH,Хө,8.0,0.0
...,...,...,...,...,...,...,...
321287,2026-03-28,Day,BUCYRUS MT4400AC #1016,EX2600-6 #121,Хө,4.0,0.0
321288,2026-03-28,Day,BUCYRUS MT4400AC #1019,EX2600-6 #121,Хө,1.0,0.0
321845,2026-03-30,Day,BUCYRUS MT4400AC #1016,EX2600-6 #121,Хө,1.0,0.0
321974,2026-03-30,Night,BUCYRUS MT4400AC #1018,EX2600-6BH#133,Хө,2.0,0.0


In [30]:
data_to_process.drop_duplicates(inplace=True)
data_to_process.to_excel("resources/产量_处理.xlsx", index=False)

In [31]:
print(df_sheet1["数量"].isnull().sum())
df_sheet1.dropna(inplace=True, subset=["数量"])
print(df_sheet1["数量"].isnull().sum())

0
0


In [32]:
df = pd.DataFrame()
df['设备名称'] = df_sheet1['挖机名称'].str.strip().unique()
df['来源'] = f"产量_{xlsx.sheet_names[0]}"
df2 = pd.DataFrame()
df2['设备名称'] = df_sheet1['矿卡名称'].str.strip().unique()
df2['来源'] = f"产量_{xlsx.sheet_names[1]}"
df = pd.concat([df, df2], ignore_index=True)
df.drop_duplicates(inplace=True)
df

,设备名称,来源
0,CAT 390D #02,产量_生产数据
1,PC 1250 #04KH,产量_生产数据
2,LIEBHERR R9350 #06,产量_生产数据
3,CAT 992K #01,产量_生产数据
4,EX5600E-6 #10,产量_生产数据
...,...,...
979,NTE240#1221,产量_运行数据
980,NTE240#1222,产量_运行数据
981,NHL NTE240AC #1221,产量_运行数据
982,NHL NTE240AC #1220,产量_运行数据


In [33]:
match_table = pd.concat([match_table, df[['设备名称', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
2107,NTE240#1221,NaN,NaN,产量_运行数据
2108,NTE240#1222,NaN,NaN,产量_运行数据
2109,NHL NTE240AC #1221,NaN,NaN,产量_运行数据
2110,NHL NTE240AC #1220,NaN,NaN,产量_运行数据


## 效率

In [34]:
# import data
file_path = 'Resources/效率_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['Sheet1']


In [ ]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)

In [36]:
df_sheet1

,日期,班次,序号 Д/д,设备种类 Техникийн төрөл,公司 Компани,应运行分钟 Ажиллах мин,应运行小时数 Ажиллах мот цаг,停车/换班/\nСул Зогсолт /Ээлж солигдох /,转移 Шилжих\nхөдөлгөөн,挖机场地推土 /清理墙壁/\nУл талбай түрүүлэх/Ханын цэвэрлэгээ/,...,未计划/故障 Төлөвлөгөөт бус/эвдрэл,待命 Бэлэн байдалд,因天气：大风暴，雨，雪Цаг агаар: Хүчтэй салхи. Бороо.Цас,扬尘：洒水车不足 Тоосжилт: Усны машин хангалтгүй,排队/装水/ Дараалалд зогсох /Ус дүүргэх/,总产量生产运行分钟 Нийт бүтээлтэй ажилласан мин,因电力原因停车。/计划/Цахилгаанаас шалтгаалсан зогсолт ./Төлөвлөсөн/,因电力原因停车。/未计划/Цахилгаанаас шалтгаалсан зогсолт /Төлөвлөгөөт бус/,总产量生产运行小时 Нийт бүтээлтэй ажилласан цаг,注释 Тайлбар
0,2019-08-01,Day,1,LIEBHERR R996B #07,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,570,NaN,NaN,9.5,NaN
1,2019-08-01,Day,2,HITACHI EX3600 #01KH,NaN,720,12,60,30,30,...,NaN,30,NaN,NaN,NaN,510,NaN,NaN,8.5,NaN
2,2019-08-01,Day,3,HITACHI EX2600E #08,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,600,NaN,NaN,10,NaN
3,2019-08-01,Day,4,HITACHI EX2600E #03KH,NaN,720,12,60,210,20,...,NaN,NaN,NaN,NaN,NaN,370,NaN,NaN,6.166667,NaN
4,2019-08-01,Day,5,LIEBHERR R9350 #06,NaN,720,12,70,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,580,NaN,NaN,9.666667,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416409,2026-04-30,Night,142,XDE120 XD#1174,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416410,2026-04-30,Night,143,XDE120 XD#1175,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416411,2026-04-30,Night,144,XDE120 XD#1176,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN
416412,2026-04-30,Night,145,XDE120 XD#1177,Normount,720,12,NaN,NaN,NaN,...,720,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN


In [39]:
df_sheet1["设备种类                             Техникийн төрөл "].value_counts()

设备种类                             Техникийн төрөл 
CAT D8T #168                                         3156
CAT D8T #169                                         3156
CAT D8T #172                                         3156
CAT D8T #170                                         3152
CAT D8T #171                                         3152
                                                     ... 
NTE240#1220                                             1
NTE240#1221                                             1
NTE240#1222                                             1
设备种类                             Техникийн төрөл        1
WT#1212                                                 1
Name: count, Length: 876, dtype: int64

In [40]:
# 查看第4列
df_sheet1['设备名称'] = df_sheet1.iloc[:, 3].str.strip()
df_sheet1['公司'] = df_sheet1.iloc[:, 4].str.strip()
df_sheet1['来源'] = f"效率_{xlsx.sheet_names[0]}"
df = df_sheet1[['设备名称','公司','来源']]
df.drop_duplicates(inplace=True)
df

,设备名称,公司,来源
0,LIEBHERR R996B #07,NaN,效率_Sheet1
1,HITACHI EX3600 #01KH,NaN,效率_Sheet1
2,HITACHI EX2600E #08,NaN,效率_Sheet1
3,HITACHI EX2600E #03KH,NaN,效率_Sheet1
4,LIEBHERR R9350 #06,NaN,效率_Sheet1
...,...,...,...
382524,Terex 60 #1201,Normount,效率_Sheet1
412504,LO#165,Normount,效率_Sheet1
412649,LONKING#165,Normount,效率_Sheet1
413063,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1


In [41]:
match_table = pd.concat([match_table, df[['设备名称', '来源', '公司']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3096,Terex 60 #1201,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


In [42]:
match_table.drop_duplicates(inplace=True, subset=['设备名称', '设备编号'])
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3084,CAT-D9GC #177,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


In [43]:
match_table.drop_duplicates(inplace=True, subset=['设备名称', '公司'])
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3084,CAT-D9GC #177,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


## 导出

In [44]:
match_table.to_excel('Resources/设备名称匹配清单.xlsx', index=False)

# 匹配

## 导入

In [2]:
# import data
table_to_match_fp = "Resources/设备名称匹配清单.xlsx"
table_to_match = pd.ExcelFile(table_to_match_fp)
print(table_to_match.sheet_names)

['Sheet1']


In [3]:
table_to_match_df = pd.read_excel(table_to_match_fp, sheet_name=0)
table_to_match_df.head()

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息


In [4]:
table_to_match_df.columns

Index(['设备名称', '设备编号', '公司', '来源'], dtype='str')

In [5]:
# import data
match_table_fp = "Resources/设备名称表.xlsx"
match_table = pd.ExcelFile(match_table_fp)
print(match_table.sheet_names)

['匹配表', '设备台账', 'Sheet1', '型号对应表', '编号分类表', '辅助表Auxiliary']


In [6]:
match_table_df = pd.read_excel(match_table_fp, sheet_name=0)
match_table_df.head()

,标准设备编号,标准设备名称,标准公司名称,标准编号前缀,编号类别,设备名称,设备编号
0,CP#0001,CHUU-COMPERESSOR CP#0001,Normount,CP,压缩机,CHUU-COMPERESSOR,CHUU001
1,CP#0001,CHUU-COMPERESSOR CP#0001,Normount,CP,压缩机,CP-0006,CHUU001
2,CP#0006,COMPRESSORS CP#0006,Normount,CP,压缩机,ATLASCOPCO CP#006,CP006
3,CP#0006,COMPRESSORS CP#0006,Normount,CP,压缩机,COMPRESSORS #006,CP0006
4,CP#0006,COMPRESSORS CP#0006,Normount,CP,压缩机,COMPRESSORS #006 /CP0005/,CP0006


In [7]:
match_table_df.columns

Index(['标准设备编号', '标准设备名称', '标准公司名称', '标准编号前缀', '编号类别', '设备名称', '设备编号'], dtype='str')

# 匹配

In [8]:
import pandas as pd
import numpy as np
import re
import difflib

# --- 辅助函数：提取数字 ---
def get_numbers(text):
    """
    根据规则提取数字：提取"#"后的数字，如果没有则提取所有连续数字
    返回一个包含且转换成整型的数字列表
    """
    if pd.isna(text):
        return []
    text = str(text)

    # 如果没有 # 或者 # 后没数字，提取所有数字
    all_nums = re.findall(r'\d+', text)
    return [int(num) for num in all_nums]

# --- 辅助函数：计算字符串相似度 ---
def get_similarity(str1, str2):
    if pd.isna(str1) or pd.isna(str2):
        return 0
    return difflib.SequenceMatcher(None, str(str1), str(str2)).ratio()

# --- 主核心匹配逻辑 ---
def match_devices(table_to_match_df, match_table_df):

    # 1. 预处理：互补填充缺失值
    table_to_match_df = table_to_match_df.copy()
    match_table_df = match_table_df.copy()

    mask_name_na = table_to_match_df['设备名称'].isna() | (table_to_match_df['设备名称'] == '')
    mask_id_na = table_to_match_df['设备编号'].isna() | (table_to_match_df['设备编号'] == '')

    table_to_match_df.loc[mask_name_na, '设备名称'] = table_to_match_df.loc[mask_name_na, '设备编号']
    table_to_match_df.loc[mask_id_na, '设备编号'] = table_to_match_df.loc[mask_id_na, '设备名称']

    # 预处理 match_table_df：提取标准设备编号中的数字，方便后续快速过滤
    match_table_df['标准编号_num'] = match_table_df['标准设备编号'].apply(
        lambda x: int(re.search(r'#(\d{4})', str(x)).group(1)) if pd.notna(x) and re.search(r'#(\d{4})', str(x)) else -1
    )

    # 准备输出的列
    results = []

    # 逐行进行匹配
    for idx, row in table_to_match_df.iterrows():
        dev_name = row['设备名称']
        dev_id = row['设备编号']

        match_result = {
            '标准设备编号': np.nan,
            '标准设备名称': np.nan,
            '标准公司名称': np.nan,
            '匹配情况': False,
            '匹配模式': ''
        }

        # --- 规则2：通过设备名称精确匹配 ---
        match_by_name = match_table_df[match_table_df['设备名称'] == dev_name]
        if not match_by_name.empty:
            best_match = match_by_name.iloc[0]
            match_result.update({'标准设备编号': best_match['标准设备编号'], '标准设备名称': best_match['标准设备名称'], '标准公司名称': best_match['标准公司名称'], '匹配情况': True, '匹配模式': '2. 设备名称精确匹配'})
            results.append(match_result)
            continue

        # --- 规则3：通过设备编号精确匹配 ---
        match_by_id = match_table_df[match_table_df['设备编号'] == dev_id]
        if not match_by_id.empty:
            best_match = match_by_id.iloc[0]
            match_result.update({'标准设备编号': best_match['标准设备编号'], '标准设备名称': best_match['标准设备名称'], '标准公司名称': best_match['标准公司名称'], '匹配情况': True, '匹配模式': '3. 设备编号精确匹配'})
            results.append(match_result)
            continue

        # 定义一个内部复用函数，用于处理数字匹配和相似度校验
        def try_num_match(nums_to_check, current_dev_name, mode_text):
            for num in nums_to_check:
                for target_num in [num, num + 1000]: # 包含原始数字和加1000的尝试
                    candidates = match_table_df[match_table_df['标准编号_num'] == target_num]

                    if len(candidates) == 1:
                        return candidates.iloc[0], f"{mode_text} (唯一匹配, 提取数值={target_num})"

                    elif len(candidates) > 1:
                        # 编号有重叠，计算名称相似度
                        candidates = candidates.copy()
                        candidates['相似度'] = candidates['设备名称'].apply(lambda x: get_similarity(current_dev_name, x))
                        best_candidate = candidates.sort_values(by='相似度', ascending=False).iloc[0]
                        return best_candidate, f"{mode_text} (多重命中取相似度最高, 提取数值={target_num})"
            return None, ""

        # --- 规则4：通过设备编号提取数字模糊匹配 ---
        id_nums = get_numbers(dev_id)
        best_match, mode_desc = try_num_match(id_nums, dev_name, "4. 设备编号数字提取匹配")
        if best_match is not None:
            match_result.update({'标准设备编号': best_match['标准设备编号'], '标准设备名称': best_match['标准设备名称'], '标准公司名称': best_match['标准公司名称'], '匹配情况': True, '匹配模式': mode_desc})
            results.append(match_result)
            continue

        # --- 规则5：通过设备名称提取数字模糊匹配 ---
        name_nums = get_numbers(dev_name)
        best_match, mode_desc = try_num_match(name_nums, dev_name, "5. 设备名称数字提取匹配")
        if best_match is not None:
            match_result.update({'标准设备编号': best_match['标准设备编号'], '标准设备名称': best_match['标准设备名称'], '标准公司名称': best_match['标准公司名称'], '匹配情况': True, '匹配模式': mode_desc})
            results.append(match_result)
            continue

        # --- 均未匹配成功 ---
        match_result['匹配模式'] = '匹配失败'
        results.append(match_result)

    # 整合结果
    result_df = pd.DataFrame(results)
    final_df = pd.concat([table_to_match_df.reset_index(drop=True), result_df], axis=1)

    # 调整列顺序
    final_columns = ['设备名称', '设备编号', '公司', '来源', '标准设备编号', '标准设备名称', '标准公司名称', '匹配情况', '匹配模式']

    return final_df[final_columns]

In [9]:

# --- 测试用例 (您可以替换为您自己的 DataFrame) ---
# df1 = pd.DataFrame(...)
# df2 = pd.DataFrame(...)
output_df = match_devices(table_to_match_df, match_table_df)
# print(output_df)

In [10]:
output_df

,设备名称,设备编号,公司,来源,标准设备编号,标准设备名称,标准公司名称,匹配情况,匹配模式
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息,HT#1056,TEREX TR100 HT#1056,Normount,True,2. 设备名称精确匹配
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息,HT#1058,TEREX TR100 HT#1058,Normount,True,2. 设备名称精确匹配
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息,HT#1059,TEREX TR100 HT#1059,Normount,True,2. 设备名称精确匹配
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息,HT#1060,TEREX TR100 HT#1060,Normount,True,2. 设备名称精确匹配
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息,HT#1061,TEREX TR100 HT#1061,Normount,True,2. 设备名称精确匹配
...,...,...,...,...,...,...,...,...,...
1661,CAT-D9GC #177,CAT-D9GC #177,Normount,效率_Sheet1,GM#0009,Generator C88D5 UNITRA GM#0009,Normount,True,"4. 设备编号数字提取匹配 (唯一匹配, 提取数值=9)"
1662,LO#165,LO#165,Normount,效率_Sheet1,NaN,NaN,NaN,False,匹配失败
1663,LONKING#165,LONKING#165,Normount,效率_Sheet1,NaN,NaN,NaN,False,匹配失败
1664,设备种类 Техникийн төрөл,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1,NaN,NaN,NaN,False,匹配失败


In [11]:
output_df["匹配情况"].value_counts()

匹配情况
True     1404
False     262
Name: count, dtype: int64

In [12]:
output_df.to_excel("resources/output.xlsx", index=False)

In [13]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz


# =========================
# 1. 基础清洗函数
# =========================
def normalize_text(x):
    """统一文本格式：空值转空串、去首尾空格、去中间多余空格、转大写"""
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r'\s+', '', x)
    return x


def fill_name_or_code(df):
    """
    缺少设备编号的用设备名称填充；
    缺少设备名称的用设备编号填充
    """
    df = df.copy()

    df["设备名称"] = df["设备名称"].astype(object)
    df["设备编号"] = df["设备编号"].astype(object)

    # 先标准化空值判断
    df["设备名称"] = df["设备名称"].replace(["", "NONE", "NAN", "NULL"], np.nan)
    df["设备编号"] = df["设备编号"].replace(["", "NONE", "NAN", "NULL"], np.nan)

    # 缺编号用名称填
    df["设备编号"] = df["设备编号"].fillna(df["设备名称"])
    # 缺名称用编号填
    df["设备名称"] = df["设备名称"].fillna(df["设备编号"])

    return df


# =========================
# 2. 数字提取函数
# =========================
def extract_number_from_code(code):
    """
    从设备编号中提取数字：
    1) 优先提取 '#' 后面的第一组数字
    2) 如果没有 '#'，则提取字符串中所有数字并拼接
    返回 int 或 None
    """
    code = normalize_text(code)
    if not code:
        return None

    # 优先取 # 后的第一组数字
    m = re.search(r'#(\d+)', code)
    if m:
        return int(m.group(1))

    # 否则提取所有数字并拼接
    nums = re.findall(r'\d+', code)
    if nums:
        return int(''.join(nums))

    return None


def extract_all_numbers_from_name(text):
    """
    从设备名称中提取所有数字组，返回整数列表
    例如：
    '1#主变-3128备用' -> [1, 3128]
    """
    text = normalize_text(text)
    if not text:
        return []

    nums = re.findall(r'\d+', text)
    if nums:
        return [int(x) for x in nums]
    return []


def extract_number_from_standard_code(std_code):
    """
    从标准设备编号中提取数字
    标准设备编号格式类似 XX#0000
    """
    std_code = normalize_text(std_code)
    if not std_code:
        return None

    m = re.search(r'#(\d+)', std_code)
    if m:
        return int(m.group(1))

    nums = re.findall(r'\d+', std_code)
    if nums:
        return int(''.join(nums))
    return None


# =========================
# 3. 名称相似度
# =========================
def calc_similarity(a, b):
    """
    名称相似度：这里用 token_sort_ratio，比较适合中文短文本 + 顺序轻微变化
    返回 0~100
    """
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0
    return fuzz.token_sort_ratio(a, b)


# =========================
# 4. 构建索引
# =========================
def build_match_indices(match_table_df):
    """
    为 match_table_df 建立精确匹配索引和数字索引
    """
    df = match_table_df.copy()

    # 标准化关键列
    for col in ["标准设备编号", "标准设备名称", "标准公司名称", "设备名称", "设备编号"]:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)
        else:
            df[col] = ""

    # 提取标准编号数字
    df["标准编号数字"] = df["标准设备编号"].apply(extract_number_from_standard_code)

    # 名称精确匹配索引
    name_index = {}
    for idx, row in df.iterrows():
        key = row["设备名称"]
        if key:
            name_index.setdefault(key, []).append(idx)

    # 编号精确匹配索引
    code_index = {}
    for idx, row in df.iterrows():
        key = row["设备编号"]
        if key:
            code_index.setdefault(key, []).append(idx)

    # 标准编号数字索引
    std_num_index = {}
    for idx, row in df.iterrows():
        key = row["标准编号数字"]
        if pd.notna(key):
            std_num_index.setdefault(int(key), []).append(idx)

    return df, name_index, code_index, std_num_index


# =========================
# 5. 候选记录选择逻辑
# =========================
def select_best_candidate(source_name, candidates_df, threshold=85):
    """
    从候选记录中选择名称最相似的一条
    返回:
    {
        "matched": bool,
        "best_score": xx,
        "best_name": xx,
        "best_row": Series or None
    }
    """
    if candidates_df.empty:
        return {
            "matched": False,
            "best_score": 0,
            "best_name": "",
            "best_row": None
        }

    best_score = -1
    best_name = ""
    best_row = None

    for _, row in candidates_df.iterrows():
        score = calc_similarity(source_name, row["标准设备名称"])
        if score > best_score:
            best_score = score
            best_name = row["标准设备名称"]
            best_row = row

    return {
        "matched": best_score >= threshold,
        "best_score": best_score if best_score >= 0 else 0,
        "best_name": best_name,
        "best_row": best_row
    }


# =========================
# 6. 主匹配函数
# =========================
def match_devices(table_to_match_df, match_table_df, similarity_threshold=85):
    # ---- 预处理待匹配表 ----
    src = table_to_match_df.copy()
    src = fill_name_or_code(src)

    # 保留原始输出列
    for col in ["设备名称", "设备编号", "公司", "来源"]:
        if col not in src.columns:
            src[col] = ""

    # 标准化源表关键字段（用于匹配）
    src["设备名称_标准化"] = src["设备名称"].apply(normalize_text)
    src["设备编号_标准化"] = src["设备编号"].apply(normalize_text)

    # ---- 构建匹配表索引 ----
    mt, name_index, code_index, std_num_index = build_match_indices(match_table_df)

    results = []

    for _, row in src.iterrows():
        raw_name = row["设备名称"]
        raw_code = row["设备编号"]
        name_std = row["设备名称_标准化"]
        code_std = row["设备编号_标准化"]

        result = {
            "设备名称": raw_name,
            "设备编号": raw_code,
            "公司": row.get("公司", ""),
            "来源": row.get("来源", ""),
            "标准设备编号": "",
            "标准设备名称": "",
            "标准公司名称": "",
            "匹配情况": False,
            "匹配模式": "",
            "最高匹配相似度": 0,
            "最高匹配名称": ""
        }

        # =========================
        # 规则1：设备名称精确匹配
        # =========================
        matched = False
        if name_std in name_index:
            candidates = mt.loc[name_index[name_std]]
            if len(candidates) == 1:
                best_row = candidates.iloc[0]
                result.update({
                    "标准设备编号": best_row["标准设备编号"],
                    "标准设备名称": best_row["标准设备名称"],
                    "标准公司名称": best_row["标准公司名称"],
                    "匹配情况": True,
                    "匹配模式": "设备名称精确匹配",
                    "最高匹配相似度": 100,
                    "最高匹配名称": best_row["标准设备名称"]
                })
                matched = True
            else:
                # 若名称重复，则选名称最相近的标准设备名称（通常相同）
                select_res = select_best_candidate(name_std, candidates, threshold=similarity_threshold)
                if select_res["matched"]:
                    best_row = select_res["best_row"]
                    result.update({
                        "标准设备编号": best_row["标准设备编号"],
                        "标准设备名称": best_row["标准设备名称"],
                        "标准公司名称": best_row["标准公司名称"],
                        "匹配情况": True,
                        "匹配模式": "设备名称精确匹配(重复候选中择优)",
                        "最高匹配相似度": select_res["best_score"],
                        "最高匹配名称": select_res["best_name"]
                    })
                    matched = True
                else:
                    result.update({
                        "匹配模式": "设备名称精确匹配失败(重复候选相似度不足)",
                        "最高匹配相似度": select_res["best_score"],
                        "最高匹配名称": select_res["best_name"]
                    })

        if matched:
            results.append(result)
            continue

        # =========================
        # 规则2：设备编号精确匹配
        # =========================
        if code_std in code_index:
            candidates = mt.loc[code_index[code_std]]
            if len(candidates) == 1:
                best_row = candidates.iloc[0]
                result.update({
                    "标准设备编号": best_row["标准设备编号"],
                    "标准设备名称": best_row["标准设备名称"],
                    "标准公司名称": best_row["标准公司名称"],
                    "匹配情况": True,
                    "匹配模式": "设备编号精确匹配",
                    "最高匹配相似度": 100,
                    "最高匹配名称": best_row["标准设备名称"]
                })
                matched = True
            else:
                select_res = select_best_candidate(name_std, candidates, threshold=similarity_threshold)
                if select_res["matched"]:
                    best_row = select_res["best_row"]
                    result.update({
                        "标准设备编号": best_row["标准设备编号"],
                        "标准设备名称": best_row["标准设备名称"],
                        "标准公司名称": best_row["标准公司名称"],
                        "匹配情况": True,
                        "匹配模式": "设备编号精确匹配(重复候选中择优)",
                        "最高匹配相似度": select_res["best_score"],
                        "最高匹配名称": select_res["best_name"]
                    })
                    matched = True
                else:
                    result.update({
                        "匹配模式": "设备编号精确匹配失败(重复候选相似度不足)",
                        "最高匹配相似度": select_res["best_score"],
                        "最高匹配名称": select_res["best_name"]
                    })

        if matched:
            results.append(result)
            continue

        # =========================
        # 规则3：设备编号提取数字 -> 标准设备编号数字匹配
        # 失败则再尝试 +1000
        # =========================
        code_num = extract_number_from_code(code_std)
        tried_nums = []
        if code_num is not None:
            tried_nums.append(code_num)
            tried_nums.append(code_num + 1000)

        best_global_score = result["最高匹配相似度"]
        best_global_name = result["最高匹配名称"]

        for num in tried_nums:
            if num not in std_num_index:
                continue

            candidates = mt.loc[std_num_index[num]]

            select_res = select_best_candidate(name_std, candidates, threshold=similarity_threshold)

            # 不管匹不匹配，都记录当前最高相似度
            if select_res["best_score"] > best_global_score:
                best_global_score = select_res["best_score"]
                best_global_name = select_res["best_name"]

            if select_res["matched"]:
                best_row = select_res["best_row"]
                result.update({
                    "标准设备编号": best_row["标准设备编号"],
                    "标准设备名称": best_row["标准设备名称"],
                    "标准公司名称": best_row["标准公司名称"],
                    "匹配情况": True,
                    "匹配模式": f"设备编号提取数字匹配({'原数字' if num == code_num else '+1000'})",
                    "最高匹配相似度": select_res["best_score"],
                    "最高匹配名称": select_res["best_name"]
                })
                matched = True
                break

        if matched:
            results.append(result)
            continue
        else:
            result["最高匹配相似度"] = best_global_score
            result["最高匹配名称"] = best_global_name

        # =========================
        # 规则4：设备名称提取数字 -> 标准设备编号数字匹配
        # 设备名称中可能有多组数字；每组数字都尝试原值和+1000
        # =========================
        name_nums = extract_all_numbers_from_name(name_std)

        candidate_num_set = []
        for n in name_nums:
            candidate_num_set.append(n)
            candidate_num_set.append(n + 1000)

        # 去重并保序
        seen = set()
        candidate_num_set = [x for x in candidate_num_set if not (x in seen or seen.add(x))]

        for num in candidate_num_set:
            if num not in std_num_index:
                continue

            candidates = mt.loc[std_num_index[num]]
            select_res = select_best_candidate(name_std, candidates, threshold=similarity_threshold)

            if select_res["best_score"] > result["最高匹配相似度"]:
                result["最高匹配相似度"] = select_res["best_score"]
                result["最高匹配名称"] = select_res["best_name"]

            if select_res["matched"]:
                best_row = select_res["best_row"]
                result.update({
                    "标准设备编号": best_row["标准设备编号"],
                    "标准设备名称": best_row["标准设备名称"],
                    "标准公司名称": best_row["标准公司名称"],
                    "匹配情况": True,
                    "匹配模式": f"设备名称提取数字匹配({'原数字' if num in name_nums else '+1000'})",
                    "最高匹配相似度": select_res["best_score"],
                    "最高匹配名称": select_res["best_name"]
                })
                matched = True
                break

        if not matched and not result["匹配模式"]:
            result["匹配模式"] = "未匹配"

        results.append(result)

    result_df = pd.DataFrame(results)

    # 输出列顺序
    output_cols = [
        "设备名称", "设备编号", "公司", "来源",
        "标准设备编号", "标准设备名称", "标准公司名称",
        "匹配情况", "匹配模式", "最高匹配相似度", "最高匹配名称"
    ]
    return result_df[output_cols]

In [14]:
result_df = match_devices(table_to_match_df, match_table_df, similarity_threshold=50)
result_df

,设备名称,设备编号,公司,来源,标准设备编号,标准设备名称,标准公司名称,匹配情况,匹配模式,最高匹配相似度,最高匹配名称
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息,HT#1056,TEREXTR100HT#1056,NORMOUNT,True,设备名称精确匹配(重复候选中择优),86.666667,TEREXTR100HT#1056
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息,HT#1058,TEREXTR100HT#1058,NORMOUNT,True,设备名称精确匹配(重复候选中择优),86.666667,TEREXTR100HT#1058
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息,HT#1059,TEREXTR100HT#1059,NORMOUNT,True,设备名称精确匹配(重复候选中择优),86.666667,TEREXTR100HT#1059
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息,HT#1060,TEREXTR100HT#1060,NORMOUNT,True,设备名称精确匹配(重复候选中择优),86.666667,TEREXTR100HT#1060
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息,HT#1061,TEREXTR100HT#1061,NORMOUNT,True,设备名称精确匹配(重复候选中择优),86.666667,TEREXTR100HT#1061
...,...,...,...,...,...,...,...,...,...,...,...
1661,CAT-D9GC #177,CAT-D9GC #177,Normount,效率_Sheet1,DZ#0177,CAT-D9GCDZ#0177,NORMOUNT,True,设备编号提取数字匹配(原数字),88.888889,CAT-D9GCDZ#0177
1662,LO#165,LO#165,Normount,效率_Sheet1,,,,False,未匹配,0.000000,
1663,LONKING#165,LONKING#165,Normount,效率_Sheet1,,,,False,未匹配,0.000000,
1664,设备种类 Техникийн төрөл,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1,,,,False,未匹配,0.000000,


In [15]:
result_df["匹配情况"].value_counts()

匹配情况
True     947
False    719
Name: count, dtype: int64

In [16]:
import pandas as pd
import re
import difflib

# --- 1. 辅助函数定义 ---

# 字符串相似度计算(0-100)
def get_similarity(s1, s2):
    if pd.isna(s1) or pd.isna(s2):
        return 0
    return difflib.SequenceMatcher(None, str(s1), str(s2)).ratio() * 100

# 提取数字规则
def extract_numbers(text):
    if pd.isna(text):
        return []
    text = str(text)
    # 如果有 #，提取 # 后面的数字
    if '#' in text:
        # 匹配 # 后面的连续数字
        matches = re.findall(r'#(\d+)', text)
        if matches:
            return [int(m) for m in matches]
    # 如果没有 #，或者 # 后面没提取到，提取所有数字片段
    matches = re.findall(r'(\d+)', text)
    return [int(m) for m in matches] if matches else []

# --- 2. 主匹配逻辑函数 ---

def match_equipment(table_to_match_df, match_table_df, threshold=85):
    # 拷贝数据避免修改原始数据
    df_obj = table_to_match_df.copy()
    df_std = match_table_df.copy()

    # 规则1: 缺失值的互相填充
    df_obj['设备名称'] = df_obj['设备名称'].fillna(df_obj['设备编号'])
    df_obj['设备编号'] = df_obj['设备编号'].fillna(df_obj['设备名称'])

    # 为标准表预先提取数字，构建 {提取数字: [标准表行数据]} 的字典以加速搜索
    std_num_dict = {}
    for idx, row in df_std.iterrows():
        std_no = row['标准设备编号']
        nums = extract_numbers(std_no)
        if nums:
            std_n = nums[0] # 标准编号一般只有一个主要数字
            if std_n not in std_num_dict:
                std_num_dict[std_n] = []
            std_num_dict[std_n].append(row)

    # 准备结果列表
    results = []

    # 逐行处理需要匹配的表
    for idx, row in df_obj.iterrows():
        obj_name = row['设备名称']
        obj_no = row['设备编号']

        # 默认结果字段
        match_info = {
            '标准设备编号': None, '标准设备名称': None, '标准公司名称': None,
            '匹配情况': False, '匹配模式': '匹配失败',
            '最高匹配相似度': 0.0, '最高匹配名称': None
        }

        # 规则2: 设备名称精确匹配
        match_by_name = df_std[df_std['设备名称'] == obj_name]
        if not match_by_name.empty:
            std_row = match_by_name.iloc[0]
            match_info.update({
                '标准设备编号': std_row['标准设备编号'], '标准设备名称': std_row['标准设备名称'],
                '标准公司名称': std_row['标准公司名称'], '匹配情况': True, '匹配模式': '规则2-名称精确匹配',
                '最高匹配相似度': 100.0, '最高匹配名称': std_row['标准设备名称']
            })
            results.append({**row.to_dict(), **match_info})
            continue

        # 规则3: 设备编号精确匹配
        match_by_no = df_std[df_std['设备编号'] == obj_no]
        if not match_by_no.empty:
            std_row = match_by_no.iloc[0]
            match_info.update({
                '标准设备编号': std_row['标准设备编号'], '标准设备名称': std_row['标准设备名称'],
                '标准公司名称': std_row['标准公司名称'], '匹配情况': True, '匹配模式': '规则3-编号精确匹配',
                '最高匹配相似度': 100.0, '最高匹配名称': std_row['标准设备名称']
            })
            results.append({**row.to_dict(), **match_info})
            continue

        # 规则4 & 5 提取数字匹配逻辑的闭包函数
        def fuzzy_match_by_numbers(text_to_extract, mode_name):
            extracted_nums = extract_numbers(text_to_extract)
            best_sim = 0.0
            best_std_row = None
            best_std_name = None

            for num in extracted_nums:
                # 尝试当前数字，如果没匹配到，尝试 + 1000
                search_nums = [num, num + 1000]

                for s_num in search_nums:
                    candidates = std_num_dict.get(s_num, [])
                    for cand in candidates:
                        sim = get_similarity(obj_name, cand['标准设备名称'])
                        if sim > best_sim:
                            best_sim = sim
                            best_std_row = cand
                            best_std_name = cand['标准设备名称']

            if best_sim >= threshold and best_std_row is not None:
                return {
                    '标准设备编号': best_std_row['标准设备编号'],
                    '标准设备名称': best_std_row['标准设备名称'],
                    '标准公司名称': best_std_row['标准公司名称'],
                    '匹配情况': True, '匹配模式': mode_name,
                    '最高匹配相似度': round(best_sim, 2), '最高匹配名称': best_std_name
                }
            elif best_std_row is not None:
                # 记录最高相似度但未成功匹配的状态
                return {
                    '最高匹配相似度': round(best_sim, 2), '最高匹配名称': best_std_name
                }
            return {}

        # 规则4: 设备编号提取数字模糊匹配
        rule4_res = fuzzy_match_by_numbers(obj_no, '规则4-设备编号模糊匹配')
        if rule4_res.get('匹配情况'):
            match_info.update(rule4_res)
            results.append({**row.to_dict(), **match_info})
            continue
        else:
            match_info.update(rule4_res) # 更新可能的最高相似度信息

        # 规则5: 设备名称提取数字模糊匹配
        rule5_res = fuzzy_match_by_numbers(obj_name, '规则5-设备名称模糊匹配')
        if rule5_res.get('匹配情况'):
            match_info.update(rule5_res)
            results.append({**row.to_dict(), **match_info})
            continue
        elif rule5_res.get('最高匹配相似度', 0) > match_info['最高匹配相似度']:
            match_info.update(rule5_res)

        # 均未匹配成功
        results.append({**row.to_dict(), **match_info})

    # 转换为DataFrame并规范输出列排序
    final_cols = ['设备名称', '设备编号', '公司', '来源', '标准设备编号', '标准设备名称',
                  '标准公司名称', '匹配情况', '匹配模式', '最高匹配相似度', '最高匹配名称']
    result_df = pd.DataFrame(results)[final_cols]

    return result_df

In [17]:
# --- 3. 运行测试（你可以用你的DataFrame替换这里的测试数据） ---
result_df = match_equipment(table_to_match_df, match_table_df,30)
result_df

,设备名称,设备编号,公司,来源,标准设备编号,标准设备名称,标准公司名称,匹配情况,匹配模式,最高匹配相似度,最高匹配名称
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息,HT#1056,TEREX TR100 HT#1056,Normount,True,规则2-名称精确匹配,100.00,TEREX TR100 HT#1056
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息,HT#1058,TEREX TR100 HT#1058,Normount,True,规则2-名称精确匹配,100.00,TEREX TR100 HT#1058
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息,HT#1059,TEREX TR100 HT#1059,Normount,True,规则2-名称精确匹配,100.00,TEREX TR100 HT#1059
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息,HT#1060,TEREX TR100 HT#1060,Normount,True,规则2-名称精确匹配,100.00,TEREX TR100 HT#1060
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息,HT#1061,TEREX TR100 HT#1061,Normount,True,规则2-名称精确匹配,100.00,TEREX TR100 HT#1061
...,...,...,...,...,...,...,...,...,...,...,...
1661,CAT-D9GC #177,CAT-D9GC #177,Normount,效率_Sheet1,DZ#0177,CAT-D9GC DZ#0177,Normount,True,规则4-设备编号模糊匹配,89.66,CAT-D9GC DZ#0177
1662,LO#165,LO#165,Normount,效率_Sheet1,NaN,NaN,None,False,匹配失败,0.00,NaN
1663,LONKING#165,LONKING#165,Normount,效率_Sheet1,NaN,NaN,None,False,匹配失败,0.00,NaN
1664,设备种类 Техникийн төрөл,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1,NaN,NaN,None,False,匹配失败,0.00,NaN


In [18]:
result_df["匹配情况"].value_counts()

匹配情况
True     1144
False     522
Name: count, dtype: int64

In [19]:
# 过滤掉“设备名称”为纯数字的行
result_df = result_df[~result_df['设备名称'].str.match(r'^\d+$')]
result_df["匹配情况"].value_counts()

匹配情况
True     1116
False     455
Name: count, dtype: int64

In [20]:
result_df.to_excel("resources/result.xlsx", index=False)